# Multi-Disease Prediction — Opsi B: Stacked Model per Penyakit

**Arsitektur**: Setiap penyakit memiliki model XGBoost sendiri.  
**Shared core features** distandarisasi lintas dataset; setiap model juga menggunakan fitur spesifik penyakitnya.  
**Inference**: Satu set input user → diteruskan ke semua model → 5 skor risiko sekaligus.

Dataset yang digunakan:
| Dataset | Target | Rows |
|---|---|---|
| Heart Disease | heart disease (0/1) | 1,025 |
| Stroke | stroke (0/1) | 5,110 |
| Diabetes (Pima) | diabetes (0/1) | 768 |
| Hypertension | hypertension High/Low | 174,982 |
| CKD | ckd/notckd | 397 |


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from scipy.io import arff
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, roc_auc_score,
                              confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(42)


## 2. Load Datasets

In [ ]:
# Sesuaikan path ke lokasi file CSV / ARFF Anda
HEART_PATH       = 'heart.csv'
STROKE_PATH      = 'healthcare-dataset-stroke-data.csv'
DIABETES_PATH    = 'diabetes.csv'
HYPER_PATH       = 'hypertension_dataset.csv'
CKD_ARFF_PATH    = 'chronic_kidney_disease_full.arff'

heart  = pd.read_csv(HEART_PATH)
stroke = pd.read_csv(STROKE_PATH)
diab   = pd.read_csv(DIABETES_PATH)
hyper  = pd.read_csv(HYPER_PATH)

# CKD: parse ARFF manual (scipy gagal karena inkonsistensi nilai nominal)
CKD_ATTRS = ['age','bp','sg','al','su','rbc','pc','pcc','ba','bgr','bu','sc',
             'sod','pot','hemo','pcv','wbcc','rbcc','htn','dm','cad','appet','pe','ane','class']
with open(CKD_ARFF_PATH, 'r') as f:
    lines = f.readlines()
data_start = next(i for i,l in enumerate(lines) if l.strip().lower() == '@data') + 1
rows = []
for l in lines[data_start:]:
    l = l.strip()
    if not l or l.startswith('%'): continue
    vals = [v.strip() for v in l.split(',')]
    if len(vals) == len(CKD_ATTRS):
        rows.append(vals)
ckd = pd.DataFrame(rows, columns=CKD_ATTRS).replace('?', np.nan)

print(f"Heart   : {heart.shape}")
print(f"Stroke  : {stroke.shape}")
print(f"Diabetes: {diab.shape}")
print(f"Hyper   : {hyper.shape}")
print(f"CKD     : {ckd.shape}")


## 3. Preprocessing per Dataset

Setiap dataset dikonversi ke skema kolom standar dengan prefix `core_*` untuk shared features,
dan kolom tambahan spesifik penyakit. Target selalu bernama `target` (0/1).


In [ ]:
# ── Heart Disease ──────────────────────────────────────────────────────────────
heart_df = pd.DataFrame({
    'age'         : heart['age'],
    'systolic_bp' : heart['trestbps'],
    'glucose'     : np.nan,
    'bmi'         : np.nan,
    'cholesterol' : heart['chol'],
    'heart_rate'  : heart['thalach'],
    'smoking'     : np.nan,
    # fitur spesifik
    'cp'          : heart['cp'],
    'fbs'         : heart['fbs'],
    'restecg'     : heart['restecg'],
    'exang'       : heart['exang'],
    'oldpeak'     : heart['oldpeak'],
    'slope'       : heart['slope'],
    'ca'          : heart['ca'],
    'thal'        : heart['thal'],
    'target'      : heart['target'],
})

# ── Stroke ─────────────────────────────────────────────────────────────────────
stroke_df = pd.DataFrame({
    'age'               : stroke['age'],
    'systolic_bp'       : np.nan,
    'glucose'           : stroke['avg_glucose_level'],
    'bmi'               : stroke['bmi'],
    'cholesterol'       : np.nan,
    'heart_rate'        : np.nan,
    'smoking'           : stroke['smoking_status'].map(
                              {'smokes':1, 'formerly smoked':0.5,
                               'never smoked':0, 'Unknown':np.nan}),
    # fitur spesifik
    'hypertension_flag' : stroke['hypertension'],
    'heart_disease_flag': stroke['heart_disease'],
    'ever_married'      : stroke['ever_married'].map({'Yes':1,'No':0}),
    'target'            : stroke['stroke'],
})

# ── Diabetes (Pima) ─────────────────────────────────────────────────────────────
# Zero pada kolom medis = implicit missing
for col in ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']:
    diab[col] = diab[col].replace(0, np.nan)

diab_df = pd.DataFrame({
    'age'         : diab['Age'],
    'systolic_bp' : diab['BloodPressure'],
    'glucose'     : diab['Glucose'],
    'bmi'         : diab['BMI'],
    'cholesterol' : np.nan,
    'heart_rate'  : np.nan,
    'smoking'     : np.nan,
    # fitur spesifik
    'pregnancies'     : diab['Pregnancies'],
    'insulin'         : diab['Insulin'],
    'skin_thickness'  : diab['SkinThickness'],
    'dpf'             : diab['DiabetesPedigreeFunction'],
    'target'          : diab['Outcome'],
})

# ── Hypertension ───────────────────────────────────────────────────────────────
# CATATAN: dataset ini fully synthetic — tidak ada korelasi antara fitur dan target.
# AUC model ~0.50. Dimasukkan untuk kelengkapan pipeline; tidak representatif klinis.
hyper_df = pd.DataFrame({
    'age'         : hyper['Age'],
    'systolic_bp' : hyper['Systolic_BP'],
    'glucose'     : hyper['Glucose'],
    'bmi'         : hyper['BMI'],
    'cholesterol' : hyper['Cholesterol'],
    'heart_rate'  : hyper['Heart_Rate'],
    'smoking'     : hyper['Smoking_Status'].map({'Current':1,'Former':0.5,'Never':0}),
    # fitur spesifik
    'diastolic_bp'    : hyper['Diastolic_BP'],
    'ldl'             : hyper['LDL'],
    'hdl'             : hyper['HDL'],
    'triglycerides'   : hyper['Triglycerides'],
    'stress_level'    : hyper['Stress_Level'],
    'salt_intake'     : hyper['Salt_Intake'],
    'alcohol_intake'  : hyper['Alcohol_Intake'],
    'sleep_duration'  : hyper['Sleep_Duration'],
    'family_history'  : hyper['Family_History'].map({'Yes':1,'No':0}),
    'physical_activity': hyper['Physical_Activity_Level'].map({'High':2,'Moderate':1,'Low':0}),
    'target'          : (hyper['Hypertension'] == 'High').astype(int),
})

# ── CKD ────────────────────────────────────────────────────────────────────────
for c in ['age','bp','bgr','bu','sc','sod','pot','hemo','pcv','wbcc','rbcc']:
    ckd[c] = pd.to_numeric(ckd[c], errors='coerce')
for c in ['sg','al','su']:
    ckd[c] = pd.to_numeric(ckd[c], errors='coerce')

cat_map = {'rbc':{'normal':0,'abnormal':1},'pc':{'normal':0,'abnormal':1},
           'pcc':{'notpresent':0,'present':1},'ba':{'notpresent':0,'present':1},
           'htn':{'no':0,'yes':1},'dm':{'no':0,'yes':1},'cad':{'no':0,'yes':1},
           'appet':{'good':0,'poor':1},'pe':{'no':0,'yes':1},'ane':{'no':0,'yes':1}}
for c, m in cat_map.items():
    ckd[c] = ckd[c].map(m)

ckd_df = pd.DataFrame({
    'age'         : ckd['age'],
    'systolic_bp' : ckd['bp'],
    'glucose'     : ckd['bgr'],
    'bmi'         : np.nan,
    'cholesterol' : np.nan,
    'heart_rate'  : np.nan,
    'smoking'     : np.nan,
    # fitur spesifik
    'specific_gravity' : ckd['sg'],
    'albumin'          : ckd['al'],
    'sugar'            : ckd['su'],
    'blood_urea'       : ckd['bu'],
    'serum_creatinine' : ckd['sc'],
    'sodium'           : ckd['sod'],
    'potassium'        : ckd['pot'],
    'hemoglobin'       : ckd['hemo'],
    'pcv'              : ckd['pcv'],
    'rbc_count'        : ckd['rbcc'],
    'hypertension_flag': ckd['htn'],
    'diabetes_flag'    : ckd['dm'],
    'anemia'           : ckd['ane'],
    'target'           : (ckd['class'] == 'ckd').astype(int),
})

print("Preprocessing selesai.")
for name, df in [('heart_disease',heart_df),('stroke',stroke_df),
                  ('diabetes',diab_df),('hypertension',hyper_df),('ckd',ckd_df)]:
    print(f"  {name:15s}: {df.shape[0]} rows, {df.shape[1]-1} features, "
          f"target positive={df['target'].mean()*100:.1f}%")


## 4. Konfigurasi Model Registry

Setiap penyakit didefinisikan dengan daftar fiturnya sendiri.  
`SMOTE` diaplikasikan otomatis jika positive class < 20% atau > 80%.


In [ ]:
DISEASE_CONFIGS = {
    'heart_disease': {
        'df'      : heart_df,
        'features': ['age','systolic_bp','cholesterol','heart_rate',
                     'cp','fbs','restecg','exang','oldpeak','slope','ca','thal'],
        'label'   : 'Heart Disease',
    },
    'stroke': {
        'df'      : stroke_df,
        'features': ['age','glucose','bmi','smoking',
                     'hypertension_flag','heart_disease_flag','ever_married'],
        'label'   : 'Stroke',
    },
    'diabetes': {
        'df'      : diab_df,
        'features': ['age','systolic_bp','glucose','bmi',
                     'pregnancies','insulin','skin_thickness','dpf'],
        'label'   : 'Diabetes',
    },
    'hypertension': {
        'df'      : hyper_df,
        'features': ['age','systolic_bp','glucose','bmi','cholesterol','heart_rate',
                     'smoking','diastolic_bp','ldl','hdl','triglycerides',
                     'stress_level','salt_intake','alcohol_intake',
                     'sleep_duration','family_history','physical_activity'],
        'label'   : 'Hypertension',
    },
    'ckd': {
        'df'      : ckd_df,
        'features': ['age','systolic_bp','glucose','specific_gravity','albumin',
                     'sugar','blood_urea','serum_creatinine','sodium','potassium',
                     'hemoglobin','pcv','rbc_count','hypertension_flag',
                     'diabetes_flag','anemia'],
        'label'   : 'CKD',
    },
}
print("Konfigurasi model terdaftar:", list(DISEASE_CONFIGS.keys()))


## 5. Training — Model per Penyakit

Pipeline tiap model:
1. Pilih fitur → drop kolom tidak relevan
2. Imputasi median untuk missing values
3. SMOTE jika class imbalance ekstrem (< 20% minority)
4. Train/test split 80/20 stratified
5. XGBoostClassifier


In [ ]:
TRAINED_MODELS = {}

for key, cfg in DISEASE_CONFIGS.items():
    df     = cfg['df']
    feats  = [f for f in cfg['features'] if f in df.columns]
    X      = df[feats].copy()
    y      = df['target'].copy()

    # Imputasi
    imp   = SimpleImputer(strategy='median')
    X_imp = imp.fit_transform(X)

    # SMOTE jika perlu
    pos_rate = y.mean()
    smote_applied = False
    if pos_rate < 0.20 or pos_rate > 0.80:
        sm = SMOTE(random_state=42)
        X_imp, y = sm.fit_resample(X_imp, y)
        smote_applied = True

    X_train, X_test, y_train, y_test = train_test_split(
        X_imp, y, test_size=0.2, random_state=42, stratify=y)

    model = XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42, verbosity=0)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    auc    = roc_auc_score(y_test, y_prob)

    TRAINED_MODELS[key] = {
        'model'         : model,
        'imputer'       : imp,
        'features'      : feats,
        'auc'           : auc,
        'X_test'        : X_test,
        'y_test'        : y_test,
        'y_pred'        : y_pred,
        'y_prob'        : y_prob,
        'smote_applied' : smote_applied,
        'label'         : cfg['label'],
    }

    print(f"[{cfg['label']:15s}] AUC={auc:.4f}  SMOTE={smote_applied}  features={len(feats)}")

print("\nTraining selesai.")


## 6. Evaluasi Model

In [ ]:
# ── Classification report per model ──────────────────────────────────────────
for key, res in TRAINED_MODELS.items():
    print(f"\n{'='*55}")
    print(f"  {res['label']}")
    print(f"{'='*55}")
    print(classification_report(res['y_test'], res['y_pred'],
                                target_names=['Negative','Positive']))


In [ ]:
# ── Confusion matrices (5 subplot) ───────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (key, res) in zip(axes, TRAINED_MODELS.items()):
    cm = confusion_matrix(res['y_test'], res['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Neg','Pos']).plot(ax=ax, colorbar=False)
    ax.set_title(f"{res['label']}\nAUC={res['auc']:.3f}")
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── ROC Curve overlay ────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00']
for (key, res), color in zip(TRAINED_MODELS.items(), colors):
    fpr, tpr, _ = roc_curve(res['y_test'], res['y_prob'])
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f"{res['label']} (AUC={res['auc']:.3f})")
ax.plot([0,1],[0,1],'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Semua Model')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Feature Importance per Model

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
for ax, (key, res) in zip(axes, TRAINED_MODELS.items()):
    fi = pd.Series(res['model'].feature_importances_, index=res['features'])
    fi.nlargest(10).sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(res['label'], fontsize=10)
    ax.set_xlabel('Importance')
plt.suptitle('Top-10 Feature Importances per Model', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Multi-Prediction Inference

Satu set input → semua model → 5 skor risiko.

### Fungsi `predict_all`


In [ ]:
def predict_all(user_input: dict, verbose: bool = True) -> dict:
    """
    Terima dict fitur dari user, teruskan ke semua model.
    Fitur yang tidak tersedia di model tertentu diisi imputer (median training).

    Returns dict {disease_key: probability_positive}
    """
    results = {}
    for key, res in TRAINED_MODELS.items():
        row   = {f: user_input.get(f, np.nan) for f in res['features']}
        X_inf = pd.DataFrame([row])
        X_imp = res['imputer'].transform(X_inf)
        prob  = res['model'].predict_proba(X_imp)[0][1]
        results[key] = prob

    if verbose:
        print(f"\n{'─'*40}")
        print(f"{'Penyakit':<20} {'Risiko':>8}  {'Bar'}")
        print(f"{'─'*40}")
        for key, prob in results.items():
            bar   = '█' * int(prob * 20)
            label = TRAINED_MODELS[key]['label']
            flag  = ' ⚠' if prob > 0.6 else ''
            print(f"{label:<20} {prob*100:>6.1f}%  {bar}{flag}")
        print(f"{'─'*40}")

    return results


In [ ]:
# ── Contoh inference: pasien berisiko tinggi ─────────────────────────────────
patient_high_risk = {
    'age'         : 62,
    'systolic_bp' : 155,
    'glucose'     : 185,
    'bmi'         : 31.2,
    'cholesterol' : 265,
    'heart_rate'  : 95,
    'smoking'     : 1,        # current smoker
}
print("Pasien A (profil berisiko tinggi):")
results_A = predict_all(patient_high_risk)


In [ ]:
# ── Contoh inference: pasien profil rendah ───────────────────────────────────
patient_low_risk = {
    'age'         : 32,
    'systolic_bp' : 115,
    'glucose'     : 88,
    'bmi'         : 22.4,
    'cholesterol' : 175,
    'heart_rate'  : 68,
    'smoking'     : 0,        # never smoked
}
print("Pasien B (profil risiko rendah):")
results_B = predict_all(patient_low_risk)


In [ ]:
# ── Visualisasi perbandingan dua pasien ─────────────────────────────────────
labels  = [TRAINED_MODELS[k]['label'] for k in TRAINED_MODELS]
vals_A  = [results_A[k] * 100 for k in TRAINED_MODELS]
vals_B  = [results_B[k] * 100 for k in TRAINED_MODELS]

x    = np.arange(len(labels))
w    = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w/2, vals_A, w, label='Pasien A (risiko tinggi)', color='#e74c3c', alpha=0.85)
ax.bar(x + w/2, vals_B, w, label='Pasien B (risiko rendah)',  color='#2ecc71', alpha=0.85)
ax.axhline(60, color='orange', ls='--', lw=1, label='Threshold 60%')
ax.set_ylabel('Probabilitas Risiko (%)')
ax.set_title('Multi-Disease Risk Comparison')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 110)
ax.legend()
plt.tight_layout()
plt.savefig('multi_prediction_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Ringkasan Performa

In [ ]:
summary = pd.DataFrame([{
    'Penyakit'        : res['label'],
    'AUC'             : round(res['auc'], 4),
    'SMOTE'           : res['smote_applied'],
    'Fitur Digunakan' : len(res['features']),
    'Catatan'         : 'Synthetic data — no signal' if key == 'hypertension' else '',
} for key, res in TRAINED_MODELS.items()])

print(summary.to_string(index=False))


## Catatan Metodologis

**Heart Disease & CKD** — AUC 1.0 menandakan kemungkinan data leakage atau dataset terlalu kecil dan mudah difit. Perlu validasi silang lebih ketat (k-fold) dan uji pada dataset eksternal.

**Stroke** — AUC ~0.99 setelah SMOTE. Perlu dicek apakah SMOTE menghasilkan generalisasi nyata atau overfitting pada pola sintetis. Rekomendasi: tambahkan strategi threshold tuning untuk menangani imbalance asli (4.9% positif).

**Diabetes** — AUC 0.83, performa paling realistis. Dataset kecil (768 baris) dengan missing values tinggi di `Insulin` dan `SkinThickness`.

**Hypertension** — Dataset fully synthetic (tidak ada korelasi statistik antara fitur dan target, AUC ~0.50). Tidak dapat digunakan untuk prediksi klinis. Perlu diganti dengan dataset nyata sebelum deployment.

**Opsi pengembangan lanjutan**:
- Ganti XGBoost dengan ensemble (LightGBM + RF + LogReg) dengan soft voting
- Tambahkan threshold calibration (Platt scaling / isotonic regression) agar probabilitas lebih terkalibrasi
- Tambahkan SHAP explanation per prediksi untuk interpretabilitas klinis
